# HDCluster demo

Python/Jupyter adaptation of `main_hdcluster.m`. The notebook loads the D31 sample data and demonstrates HDCluster without denoising, with default denoising, and with a custom `beta` value.

Cluster labels start at 1; when denoising is enabled, noise points are labeled `-1`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from hdcluster import hdcluster

ImportError: bad magic number in 'hdcluster': b'\xf3\r\r\n'

In [ ]:
data_name = "D31"
data_path = Path("data") / f"{data_name}.txt"

dataset = np.loadtxt(data_path, skiprows=1)
ground_truth_labels = dataset[:, -1].astype(int)
points = dataset[:, :2]

print(f"Loaded {len(points):,} points from {data_path}")
print(f"Dimensions: {points.shape[1]}; ground-truth clusters: {np.unique(ground_truth_labels).size}")

In [ ]:
def plot_clusters(points, labels, title):
    """Plot labeled 2D or 3D spatial data in the style of the MATLAB demo."""
    if points.shape[1] == 2:
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(points[:, 0], points[:, 1], s=6, c=labels, cmap="jet")
        ax.set(xlabel="X", ylabel="Y")
        ax.set_aspect("equal", adjustable="box")
    elif points.shape[1] == 3:
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection="3d")
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], s=6, c=labels, cmap="jet")
        ax.set(xlabel="X", ylabel="Y", zlabel="Z")
        ax.set_box_aspect(np.ptp(points, axis=0))
    else:
        raise ValueError("Plotting is supported only for 2D or 3D data.")

    ax.set_title(title)
    fig.tight_layout()
    plt.show()


def cluster_summary(labels):
    cluster_count = np.unique(labels[labels > 0]).size
    noise_count = np.count_nonzero(labels == -1)
    return cluster_count, noise_count


plot_clusters(
    points,
    ground_truth_labels,
    f'Ground truth data\nDataset "{data_name}" — {np.unique(ground_truth_labels).size} clusters',
)

## Run HDCluster without denoising

In [ ]:
mth = 1.0
centers, cluster_ids = hdcluster(points, mth, workers=-1)
cluster_count, noise_count = cluster_summary(cluster_ids)

print(f"Clusters: {cluster_count}; noise points: {noise_count}; centers: {len(centers)}")
plot_clusters(
    points,
    cluster_ids,
    f'Method: HDCluster — merging threshold = {mth:g}\nDataset "{data_name}" — {cluster_count} clusters',
)

## Run HDCluster with default denoising

As in the MATLAB script, this run enables denoising and uses HDCluster's default `beta=0.2`.

In [ ]:
mth = 1
centers, cluster_ids = hdcluster(points, mth, is_noise=True, workers=-1)
cluster_count, noise_count = cluster_summary(cluster_ids)

print(f"Clusters: {cluster_count}; noise points: {noise_count}; centers: {len(centers)}")
plot_clusters(
    points,
    cluster_ids,
    f'Method: HDCluster with denoising — merging threshold = {mth:g}\nDataset "{data_name}" — {cluster_count} clusters',
)

## Run HDCluster with a custom denoising parameter

In [ ]:
mth = 1
beta = 1.4
centers, cluster_ids = hdcluster(points, mth, is_noise=True, beta=beta, workers=-1)
cluster_count, noise_count = cluster_summary(cluster_ids)

print(f"Clusters: {cluster_count}; noise points: {noise_count}; centers: {len(centers)}")
plot_clusters(
    points,
    cluster_ids,
    f'Method: HDCluster with denoising — merging threshold = {mth:g}, beta = {beta:g}\nDataset "{data_name}" — {cluster_count} clusters',
)